# AgriSense — Classification Baseline

This notebook establishes the first disease-classification baseline
for AgriSense.

The goal is to determine how well a pretrained image classification
model can classify the 115 disease classes using the original plant
images.

This baseline will provide a reference point for later improvements
such as class-weighted training, sampling strategies, and
segmentation-aware approaches.

## Model

- Architecture: ResNet-18
- Initialization: ImageNet pretrained weights
- Input size: 224 × 224
- Number of classes: 115
- Training data: AgriSense cleaned Training split
- Validation data: AgriSense cleaned Validation split
- Test data: AgriSense cleaned Test split

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from tqdm.auto import tqdm

from PIL import Image

from torchvision import models , transforms

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

In [3]:
#loading the preprocessing objects

project_root = Path.cwd().parent

processed_dir = project_root / "data" / "processed"

clean_metadata_path = (
    processed_dir / "agrisense_metadata_clean.csv"
)

df_clean = pd.read_csv(clean_metadata_path)


In [4]:
disease_classes = sorted(df_clean["Disease"].unique())

disease_to_id = {
    disease: idx
    for idx, disease in enumerate(disease_classes)
}

id_to_disease = {
    idx: disease
    for disease, idx in disease_to_id.items()
}

df_clean["disease_id"] = df_clean["Disease"].map(disease_to_id)

print("Number of disease classes:", len(disease_classes))
print("Missing disease IDs:", df_clean["disease_id"].isna().sum())
print("Unique disease IDs:", df_clean["disease_id"].nunique())

Number of disease classes: 115
Missing disease IDs: 0
Unique disease IDs: 115


In [5]:
print("Dataset shape:", df_clean.shape)
print("Disease classes:", df_clean["disease_id"].nunique())

Dataset shape: (7399, 12)
Disease classes: 115


In [6]:
# Recreating the split DataFrames

train_df = df_clean[
    df_clean["Split"] == "Training"
].copy()

val_df = df_clean[
    df_clean["Split"] == "Validation"
].copy()

test_df = df_clean[
    df_clean["Split"] == "Test"
].copy()

print("Training:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

Training: 5084
Validation: 811
Test: 1504


## Reusing my Dataset and transforms

In [7]:
# 1

class ResizeWithPadding:
    def __init__(self, size=(224, 224)):
        self.size = size

    def __call__(self, image):
        target_width, target_height = self.size

        scale = min(
            target_width / image.width,
            target_height / image.height
        )

        new_width = int(image.width * scale)
        new_height = int(image.height * scale)

        image = image.resize(
            (new_width, new_height),
            Image.Resampling.LANCZOS
        )

        canvas = Image.new(
            "RGB",
            self.size,
            (0, 0, 0)
        )

        left = (target_width - new_width) // 2
        top = (target_height - new_height) // 2

        canvas.paste(image, (left, top))

        return canvas

In [8]:
# 2 

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    ResizeWithPadding((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])

val_test_transform = transforms.Compose([
    ResizeWithPadding((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])

In [9]:
# 3

class AgriSenseDataset(Dataset):

    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        image_path = Path(row["actual_image_path"])

        image = Image.open(image_path).convert("RGB")
        label = int(row["disease_id"])

        if self.transform:
            image = self.transform(image)
    
        return image, label

In [10]:
def find_actual_image_path(row):
    expected_path = image_root / row["Name"]

    if expected_path.exists():
        return expected_path

    matches = list(image_root.rglob(row["Name"]))

    if len(matches) == 1:
        return matches[0]

    return None


In [13]:
image_root = (
    project_root
    / "data"
    / "raw"
    / "PlantSeg"
    / "images"
)

df_clean["actual_image_path"] = df_clean.apply(
    find_actual_image_path,
    axis=1
)

In [14]:
print(
    "Images successfully located:",
    df_clean["actual_image_path"].notna().sum()
)

print(
    "Images not located:",
    df_clean["actual_image_path"].isna().sum()
)

Images successfully located: 7399
Images not located: 0


In [15]:
train_df = df_clean[
    df_clean["Split"] == "Training"
].copy()

val_df = df_clean[
    df_clean["Split"] == "Validation"
].copy()

test_df = df_clean[
    df_clean["Split"] == "Test"
].copy()

### creating dataset and loaders

In [16]:
train_dataset = AgriSenseDataset(
    train_df,
    transform=train_transform
)

val_dataset = AgriSenseDataset(
    val_df,
    transform=val_test_transform
)

test_dataset = AgriSenseDataset(
    test_df,
    transform=val_test_transform
)

In [26]:
BATCH_SIZE = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

In [27]:
# checking the device

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

Device: cpu


In [28]:
#loading pretrained RestNet-18

weights = models.ResNet18_Weights.DEFAULT

model = models.resnet18(weights=weights)

print(model.fc)

Linear(in_features=512, out_features=1000, bias=True)


In [29]:
#replacing final layers

num_classes = df_clean["disease_id"].nunique()

model.fc = nn.Linear(
    model.fc.in_features,
    num_classes
)

model = model.to(device)

print(model.fc)

Linear(in_features=512, out_features=115, bias=True)


In [30]:
#for the first baseline we freeze the backbone

for parameter in model.parameters():
    parameter.requires_grad = False

for parameter in model.fc.parameters():
    parameter.requires_grad = True

In [31]:
trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

print("Trainable parameters:", trainable_parameters)
print("Total parameters:", total_parameters)

Trainable parameters: 58995
Total parameters: 11235507


In [32]:
#loss and optimizer

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.fc.parameters(),
    lr=0.001
)

In [33]:
images, labels = next(iter(train_loader))

print("Input batch:", images.shape)
print("Labels:", labels.shape)

images = images.to(device)
labels = labels.to(device)

with torch.no_grad():
    outputs = model(images)

print("Model output:", outputs.shape)

Input batch: torch.Size([16, 3, 224, 224])
Labels: torch.Size([16])
Model output: torch.Size([16, 115])


In [37]:
EPOCHS = 1

best_val_loss = float("inf")
best_model_state = None

In [ ]:
# training + validation loop

train_history = []
val_history = []

# AgriSense - ResNet18 Baseline Training

# 1. Training configuration

EPOCHS = 5
BATCH_SIZE = 16

print("Epochs:", EPOCHS)
print("Batch size:", BATCH_SIZE)
print("Device:", device)


# 2. Recreate DataLoaders with the stable batch size

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print("\nDataLoaders created successfully.")
print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))


# 3. Create checkpoint directory

checkpoint_dir = project_root / "models" / "checkpoints"

checkpoint_dir.mkdir(
    parents=True,
    exist_ok=True
)

latest_checkpoint_path = (
    checkpoint_dir / "resnet18_baseline_latest.pth"
)

best_checkpoint_path = (
    checkpoint_dir / "resnet18_baseline_best.pth"
)

print("\nCheckpoint directory:")
print(checkpoint_dir)


# 4. Training history

train_history = []
val_history = []

best_val_loss = float("inf")


# 5. Training loop

for epoch in range(EPOCHS):

    print("\n" + "=" * 60)
    print(f"Starting Epoch {epoch + 1}/{EPOCHS}")
    print("=" * 60)


    # TRAINING

    model.train()

    running_train_loss = 0.0

    train_predictions = []
    train_targets = []


    train_progress = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/{EPOCHS}"
    )


    for images, labels in train_progress:

        # Move data to CPU/GPU
        images = images.to(device)
        labels = labels.to(device)


        # Clear previous gradients
        optimizer.zero_grad()


        # Forward pass
        outputs = model(images)


        # Calculate loss
        loss = criterion(outputs, labels)


        # Backpropagation
        loss.backward()


        # Update trainable parameters
        optimizer.step()


        # Accumulate loss
        running_train_loss += (
            loss.item() * images.size(0)
        )


        # Get predicted class
        predictions = torch.argmax(
            outputs,
            dim=1
        )


        # Store predictions and true labels
        train_predictions.extend(
            predictions.cpu().numpy()
        )

        train_targets.extend(
            labels.cpu().numpy()
        )


    # Calculate average training loss
    train_loss = (
        running_train_loss
        / len(train_loader.dataset)
    )


    # Calculate training accuracy
    train_accuracy = accuracy_score(
        train_targets,
        train_predictions
    )


    # Calculate training Macro F1
    train_f1 = f1_score(
        train_targets,
        train_predictions,
        average="macro",
        zero_division=0
    )


    # VALIDATION

    model.eval()

    running_val_loss = 0.0

    val_predictions = []
    val_targets = []


    with torch.no_grad():

        val_progress = tqdm(
            val_loader,
            desc="Validation"
        )


        for images, labels in val_progress:

            images = images.to(device)
            labels = labels.to(device)


            # Forward pass
            outputs = model(images)


            # Validation loss
            loss = criterion(
                outputs,
                labels
            )


            # Accumulate loss
            running_val_loss += (
                loss.item() * images.size(0)
            )


            # Predicted class
            predictions = torch.argmax(
                outputs,
                dim=1
            )


            # Store predictions and true labels
            val_predictions.extend(
                predictions.cpu().numpy()
            )

            val_targets.extend(
                labels.cpu().numpy()
            )


    # Calculate average validation loss
    val_loss = (
        running_val_loss
        / len(val_loader.dataset)
    )


    # Calculate validation accuracy
    val_accuracy = accuracy_score(
        val_targets,
        val_predictions
    )


    # Calculate validation Macro F1
    val_f1 = f1_score(
        val_targets,
        val_predictions,
        average="macro",
        zero_division=0
    )


    # SAVE METRICS TO HISTORY

    train_history.append({
        "epoch": epoch + 1,
        "loss": train_loss,
        "accuracy": train_accuracy,
        "macro_f1": train_f1
    })


    val_history.append({
        "epoch": epoch + 1,
        "loss": val_loss,
        "accuracy": val_accuracy,
        "macro_f1": val_f1
    })


    # SAVE LATEST CHECKPOINT

    torch.save(
        {
            "epoch": epoch + 1,

            "model_state_dict":
                model.state_dict(),

            "optimizer_state_dict":
                optimizer.state_dict(),

            "train_loss":
                train_loss,

            "train_accuracy":
                train_accuracy,

            "train_macro_f1":
                train_f1,

            "val_loss":
                val_loss,

            "val_accuracy":
                val_accuracy,

            "val_macro_f1":
                val_f1,

            "num_classes":
                num_classes,

            "disease_to_id":
                disease_to_id
        },
        latest_checkpoint_path
    )


    # SAVE BEST CHECKPOINT

    if val_loss < best_val_loss:

        best_val_loss = val_loss

        torch.save(
            {
                "epoch": epoch + 1,

                "model_state_dict":
                    model.state_dict(),

                "optimizer_state_dict":
                    optimizer.state_dict(),

                "val_loss":
                    val_loss,

                "val_accuracy":
                    val_accuracy,

                "val_macro_f1":
                    val_f1,

                "num_classes":
                    num_classes,

                "disease_to_id":
                    disease_to_id
            },
            best_checkpoint_path
        )

        best_message = "  ✓ New best model saved!"

    else:

        best_message = ""


    # PRINT EPOCH RESULTS

    print("\nEpoch Results")
    print("-" * 60)

    print(
        f"Train Loss:     {train_loss:.4f}"
    )

    print(
        f"Train Accuracy: {train_accuracy:.4f}"
    )

    print(
        f"Train Macro F1: {train_f1:.4f}"
    )

    print()

    print(
        f"Val Loss:       {val_loss:.4f}"
    )

    print(
        f"Val Accuracy:   {val_accuracy:.4f}"
    )

    print(
        f"Val Macro F1:   {val_f1:.4f}"
    )

    print()

    print(
        f"Latest checkpoint saved."
    )

    if best_message:
        print(best_message)

# 6. Training completed

print("\n" + "=" * 60)
print("TRAINING COMPLETE")
print("=" * 60)

print(
    f"Best Validation Loss: {best_val_loss:.4f}"
)

print(
    "\nLatest checkpoint:"
)

print(latest_checkpoint_path)

print(
    "\nBest checkpoint:"
)

print(best_checkpoint_path)

Validation: 100%|██████████| 51/51 [01:09<00:00,  1.37s/it]

Epoch 1/1 | Train Loss: 3.8648 | Train Acc: 0.1682 | Train Macro F1: 0.0827 | Val Loss: 3.0346 | Val Acc: 0.3095 | Val Macro F1: 0.1577
